# Data Analysis Agent (Google Colab)

Run this notebook in Google Colab to launch the Web UI, then upload/drop a CSV in the browser.

## Quick Start (Colab Only)
1. Open this notebook in Google Colab.
2. Ensure the project files are available in the Colab runtime (for example, via Drive mount or upload).
3. Run Cell 2 (setup helpers).
4. Run Cell 3 (start server + clickable URL).
5. Open the generated Colab proxy URL, upload CSV, type question, and click Analyze Dataset.
6. (Optional) Run Cell 4 to stop the server.

Note: This launcher is intentionally Colab-only and will stop with an error outside Colab.

In [ ]:
import json
import os
import sys
import subprocess
import time
from getpass import getpass
from pathlib import Path
from urllib.request import urlopen
from urllib.error import URLError

from IPython.display import HTML, display

# Colab-only runtime guard
try:
    from google.colab import output as colab_output  # type: ignore
except Exception as e:
    raise RuntimeError(
        "This notebook launcher is Colab-only. Open it in Google Colab to continue. "
        f"Details: {e}"
    )

PORT = 8000
HOST = "0.0.0.0"
LOCAL_HEALTH_URL = f"http://127.0.0.1:{PORT}"

def _is_project_root(path: Path) -> bool:
    return (path / "app.py").exists() and (path / "static").exists()

def _scan_for_project_root(base: Path, max_depth: int = 5) -> Path | None:
    if not base.exists() or not base.is_dir():
        return None

    base_depth = len(base.parts)
    for root, dirs, files in os.walk(base):
        root_path = Path(root)
        current_depth = len(root_path.parts) - base_depth
        if current_depth > max_depth:
            dirs[:] = []
            continue

        if "app.py" in files and (root_path / "static").exists():
            return root_path

    return None

def _find_project_root(start: Path) -> Path:
    # 1) Explicit override if user has a known location in Colab runtime.
    hint = os.getenv("PROJECT_ROOT_HINT", "").strip()
    if hint:
        hint_path = Path(hint).expanduser()
        if _is_project_root(hint_path):
            return hint_path

    # 2) Check cwd and all parents first.
    for candidate in [start, *start.parents]:
        if _is_project_root(candidate):
            return candidate

    # 3) Search common Colab locations.
    common_bases = [
        Path("/content"),
        Path("/content/drive/MyDrive"),
        Path("/content/drive/MyDrive/Colab Notebooks"),
    ]
    for base in common_bases:
        found = _scan_for_project_root(base, max_depth=6)
        if found is not None:
            return found

    raise FileNotFoundError(
        "Could not locate project root containing app.py and static/.\n"
        "Set PROJECT_ROOT_HINT to your project folder, or place the project under /content.\n"
        "Example setup in Colab:\n"
        "  from google.colab import drive\n"
        "  drive.mount('/content/drive')\n"
        "  %cd /content/drive/MyDrive/<your-project-folder>"
    )

def _wait_for_server(url: str, timeout_seconds: int = 30) -> bool:
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        try:
            with urlopen(f"{url}/health", timeout=2) as response:
                if response.status == 200:
                    return True
        except URLError:
            pass
        time.sleep(1)
    return False

def _get_health(url: str):
    try:
        with urlopen(f"{url}/health", timeout=3) as response:
            if response.status == 200:
                return json.loads(response.read().decode("utf-8"))
    except Exception:
        return None
    return None

def _resolve_colab_url() -> str:
    proxied_url = colab_output.eval_js(f"google.colab.kernel.proxyPort({PORT})")
    if not isinstance(proxied_url, str) or not proxied_url.strip():
        raise RuntimeError("Failed to create a Colab proxy URL for port 8000.")
    return proxied_url

PROJECT_ROOT = _find_project_root(Path.cwd())
SERVER_PROCESS = None

print(f"Project root: {PROJECT_ROOT}")
print("Runtime: Google Colab")
print(f"Server bind host: {HOST}:{PORT}")
print(f"Local health URL: {LOCAL_HEALTH_URL}")

RuntimeError: This notebook launcher is Colab-only. Open it in Google Colab to continue. Details: No module named 'google.colab'

In [ ]:
# Start Web UI server (Colab)
is_running = SERVER_PROCESS is not None and SERVER_PROCESS.poll() is None

if is_running:
    health = _get_health(LOCAL_HEALTH_URL)
    if health and health.get("agent_initialized") is True:
        print(f"Server is already running and healthy at {LOCAL_HEALTH_URL}")
    else:
        print("A server is running but the agent is not initialized. Restarting with API key...")
        SERVER_PROCESS.terminate()
        try:
            SERVER_PROCESS.wait(timeout=10)
        except Exception:
            SERVER_PROCESS.kill()
            SERVER_PROCESS.wait(timeout=5)
        is_running = False

if not is_running:
    api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
    if not api_key:
        api_key = getpass("Enter Gemini API key for Web UI: ").strip()
        if not api_key:
            raise ValueError("Gemini API key is required to start analysis server.")

    server_env = os.environ.copy()
    server_env["GEMINI_API_KEY"] = api_key
    server_env["GOOGLE_API_KEY"] = api_key

    SERVER_PROCESS = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "app:app", "--host", HOST, "--port", str(PORT)],
        cwd=str(PROJECT_ROOT),
        env=server_env,
    )

    if not _wait_for_server(LOCAL_HEALTH_URL, timeout_seconds=30):
        raise RuntimeError("Server did not become ready in time. Check notebook output for errors.")

    health = _get_health(LOCAL_HEALTH_URL)
    if not health or health.get("agent_initialized") is not True:
        status = health.get("status") if isinstance(health, dict) else "unknown"
        detail = health.get("detail") if isinstance(health, dict) else "unknown"
        raise RuntimeError(
            f"Server is reachable but agent initialization failed (status={status}). Detail: {detail}"
        )

BASE_URL = _resolve_colab_url()
display(HTML(f'<a href="{BASE_URL}" target="_blank">Open Data Analysis Web UI (Colab)</a>'))
print(f"Web UI ready: {BASE_URL}")
print("Colab mode: using proxy URL for browser access.")
print("Now upload/drop a CSV in the browser and click Analyze Dataset.")

Web UI ready: http://127.0.0.1:8000
Local mode: using localhost URL.
Now upload/drop a CSV in the browser and click Analyze Dataset.


In [ ]:
# Optional: stop Web UI server
STOP_SERVER_NOW = False

if STOP_SERVER_NOW and SERVER_PROCESS is not None and SERVER_PROCESS.poll() is None:
    SERVER_PROCESS.terminate()
    SERVER_PROCESS.wait(timeout=10)
    print("Web UI server stopped.")
elif STOP_SERVER_NOW:
    print("No running Web UI server found.")
else:
    print("Stop skipped. Set STOP_SERVER_NOW = True to stop the server.")

Web UI server stopped.
